# Exact Minimum-Error Sequential Discrimination

This notebook runs the exhaustive-search optimal algorithm on the generated 1D and 2D lite benchmarks. It displays five candidate plots and saves detailed results, CSV summaries, and figures under `results/exhaustive_search_notebook/`.

The default pilot uses one realization from every field-family, $K$, and $M$ cell. This includes $K=3,4,5$ and $M=3,4$ while keeping the notebook practical to run interactively.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
MODULE_DIRECTORY = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
if str(MODULE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(MODULE_DIRECTORY))

from notebook_analysis import (
    make_candidate_plots,
    run_dataset,
    run_summary,
)

## Run settings

Set `RUN_ALL_REPLICATES = True` to evaluate all 3,000 cases in each dataset. The full run can take a long time because the decision-tree search grows quickly with $K$. The sensor-budget cap of 12 limits runtime but does not change the mathematical problem definition.

In [ ]:
DATASETS = ["lite_benchmark_1d", "lite_benchmark_2d"]
RUN_ALL_REPLICATES = False
REPLICATES_PER_CELL = None if RUN_ALL_REPLICATES else 1
RESULTS_ROOT = MODULE_DIRECTORY / "results" / "exhaustive_search_notebook"

print("Run mode:", "all 3,000 cases" if RUN_ALL_REPLICATES else "pilot: 1 replicate per cell")
print("Results directory:", RESULTS_ROOT)

## Run the optimal algorithm

For every selected case and sensor budget, the code finds the optimal sensor allocation, pair locations, adaptive decision tree, and final error probability.

In [ ]:
records_by_dataset = {}
for dataset in DATASETS:
    display(Markdown(f"### Running `{dataset}`"))
    records_by_dataset[dataset] = run_dataset(
        dataset,
        replicates_per_cell=REPLICATES_PER_CELL,
        results_root=RESULTS_ROOT,
    )

## Run summary

In [ ]:
summary_lines = [
    "| Dataset | Cases | Solver runs | K values | M values | Mean error | Total seconds | Longest run |",
    "|---|---:|---:|---|---|---:|---:|---:|",
]
for dataset, records in records_by_dataset.items():
    summary = run_summary(records)
    summary_lines.append(
        f"| {dataset} | {summary['cases']} | {summary['runs']} | "
        f"{summary['K values']} | {summary['M values']} | "
        f"{summary['mean error']:.4f} | {summary['total seconds']:.2f} | "
        f"{summary['maximum seconds']:.2f} |"
    )
display(Markdown("\n".join(summary_lines)))

## Candidate plots

1. **Error versus sensor budget** is the strongest primary performance plot.
2. **Error by field family** shows whether performance changes across Gaussian, quadratic, and sinusoidal fields.
3. **Error heatmap over $K$ and $M$** summarizes problem-size effects.
4. **Runtime scaling** shows the computational cost of exhaustive search.
5. **Representative case curves** provide an intuitive example but should not replace aggregate results.

In [ ]:
for dataset, records in records_by_dataset.items():
    display(Markdown(f"## Plots for `{dataset}`"))
    for title, figure in make_candidate_plots(
        dataset, records, results_root=RESULTS_ROOT
    ):
        display(Markdown(f"### {title}"))
        display(figure)
        plt.close(figure)

## Saved files

Each dataset has the following structure:

```text
results/exhaustive_search_notebook/<dataset>/
├── tables/
│   ├── results.json
│   └── summary.csv
└── figures/
    ├── error_vs_budget.*
    ├── error_by_field_family.*
    ├── error_heatmap_k_m.*
    ├── runtime_scaling.*
    └── representative_case_curves.*
```

Figures are saved as 300-dpi PNG and vector PDF files.